[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GeoLarryLai/pyRockImageLab/blob/main/pyRockImageLab_1_core_photo_processing.ipynb)


# pyRockImageLab-01 — Core Photo Processing

Two steps: **mark** corners in the browser app, then **re-create** the images here (or in batch).

- **Crop core-box photo**: mark the corners of each core section on a box photo → one straightened image per section, named by depth (`{UWI}_{start}-{end}.tif/.png`, 300 dpi).
- **Stitch close-up photos**: choose the close-ups of one section, trim each one, join them → one long high-resolution image (`{UWI}_{start}-{end}_hr.tif/.png`, 600 dpi).

All choices are saved in `_picks.json` inside each depth folder. The cells below replay them without any pop-up windows, so they also run on Colab or a server.

## Setup

In [ ]:
# %pip install --quiet pyrockimagelab pillow-heif

In [ ]:
from pathlib import Path
import pyrockimagelab as pril
from pyrockimagelab import (DEFAULT_HR_PHOTO_RANGES, N_SEGMENTS_DEFAULT, PICKS_FILENAME, list_source_images,
                            load_picks, process_folder, process_hr_segments, process_lr_segments,
                            release_working_memory, save_picks)
print(f"pyrockimagelab {pril.__version__}")

## Config

Point `FOLDER` at one depth-range folder: `{project}/{UWI}/{start-end}/`. This repo does not include your photos.

In [ ]:
FOLDER = Path("example_data/ExampleCore/2563-2572").resolve()
assert FOLDER.is_dir(), f"Folder not found: {FOLDER}"
UWI = FOLDER.parent.name            # well folder name, used in output names
PROJECT = FOLDER.parents[1]         # the folder you open in the app

N_SEGMENTS = N_SEGMENTS_DEFAULT                   # core sections per box photo (typically 3)
HR_PHOTO_RANGES = dict(DEFAULT_HR_PHOTO_RANGES)   # {section: (first, last+1)} photo numbers, used only if none saved
print(FOLDER, "| UWI =", UWI)

## Step 1 — Mark corners in the app

Opens `pyrockimagelab-app` on the project folder (stop it with `app.terminate()`). In the browser choose a tool and click this folder. Optional: pre-fill the close-up photo groups from `HR_PHOTO_RANGES` first.

In [ ]:
picks = load_picks(FOLDER)
sources = list_source_images(FOLDER)
if not picks.get("hr_assignments"):
    picks["n_segments"] = int(N_SEGMENTS)
    picks["hr_assignments"] = {f"seg{k}": list(range(lo, min(hi, len(sources))))
                               for k, (lo, hi) in HR_PHOTO_RANGES.items() if lo < len(sources)}
    save_picks(FOLDER, picks)
print(f"{len(sources)} photos | saved: box photo={picks.get('box_photo')}, "
      f"sections marked={len(picks['box_quads'])}, close-up groups={picks['hr_assignments']}")

# import subprocess, sys
# app = subprocess.Popen([sys.executable, "-m", "pyrockimagelab.webapp", str(PROJECT)])

## Step 2 — Re-create the box-photo section images (`process_lr_segments`)

In [ ]:
picks = load_picks(FOLDER)
if picks.get("box_photo"):
    saved, picks, _ = process_lr_segments(FOLDER, FOLDER / picks["box_photo"], UWI, picks=picks)
    print(f"Section images saved: {saved}")
else:
    print("No box photo chosen yet — use the app first.")

## Step 3 — Re-create the stitched high-resolution images (`process_hr_segments`)

Uses the saved photo groups, trim corners, matching settings and final corners.

In [ ]:
hr_saved, _ = process_hr_segments(FOLDER, UWI, picks)
print(f"Stitched images saved: {hr_saved}")
release_working_memory()

### Optional — whole folder or whole project in one go

`process_folder` runs Steps 2–3. For every folder under a project root, use the command line: `pyrockimagelab --root /path/to/project`.

In [ ]:
# lr_saved, hr_saved, _ = process_folder(FOLDER, UWI)